# Xiangqi AI — Bronze Ingestion: CCPD / Alpha-Beta

Mặc định: **CCPD → Master Matches → World Mind Sports Games**, 595 PGN trong thư mục này tại thời điểm kiểm tra. Notebook tải bằng Git sparse checkout, ingest raw PGN vào Bronze Drive, ghi SHA-256/manifest và giữ LICENSE + README.md.

- Sample 20 ván trước, sau đó chuyển CCPD_MODE thành "full" để lấy tất cả.
- Bronze KHÔNG chuyển PGN thành JSONL hoặc chạy Silver.
- Nguồn: https://github.com/Yvonne761/Chinese-Chess-Practical-Dataset
- Attribution: Yu-Han Tseng và Bo-Nian Chen (2026); license CC BY 4.0.
- SOURCE_CHOICE = "alphabeta" để dùng dữ liệu tự sinh từ Drive như cũ.

## 1. Mount Google Drive và cấu hình

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/XiangqiAI")
BRONZE_ROOT = DRIVE_PROJECT / "data" / "01_bronze"
BACKEND_REPO = Path("/content/ML-Advance-Group")
BACKEND_DIR = BACKEND_REPO / "project" / "backend"

SOURCE_CHOICE = "ccpd"   # "ccpd" or "alphabeta"
CCPD_MODE = "sample"     # "sample" (20 PGN) or "full" (all PGN)
CCPD_SAMPLE_SIZE = 20
CCPD_SEED = 42
BACKEND_BRANCH = "main"

if SOURCE_CHOICE not in {"ccpd", "alphabeta"} or CCPD_MODE not in {"sample", "full"}:
    raise ValueError("Invalid dataset configuration")
if CCPD_SAMPLE_SIZE < 1:
    raise ValueError("CCPD_SAMPLE_SIZE must be positive")
print("Bronze root:", BRONZE_ROOT)

## 2. Clone backend main và cài package

Sử dụng lại BronzeIngester từ scripts.ingest_bronze; không viết ETL trùng lặp trong Colab.

In [ ]:
import subprocess
import sys

if not BACKEND_REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BACKEND_BRANCH,
        "https://github.com/Ko4rut/ML-Advance-Group.git", str(BACKEND_REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(BACKEND_REPO), "fetch", "origin", BACKEND_BRANCH], check=True)
    subprocess.run(["git", "-C", str(BACKEND_REPO), "switch", BACKEND_BRANCH], check=True)
    subprocess.run([
        "git", "-C", str(BACKEND_REPO), "pull", "--ff-only", "origin", BACKEND_BRANCH
    ], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND_DIR)
], check=True)
if not (BACKEND_DIR / "scripts/ingest_bronze.py").is_file():
    raise FileNotFoundError("Missing scripts/ingest_bronze.py")
print("Backend revision:", subprocess.check_output([
    "git", "-C", str(BACKEND_REPO), "rev-parse", "--short", "HEAD"
], text=True).strip())

## 3. Tải CCPD (chỉ nhánh 595 PGN đã chọn)

Clone với bộ lọc blob và sparse checkout để tránh tải cả 58k bản ghi của repository. Lưu tài liệu license và README để đảm bảo khả năng truy xuất nguồn. Sample dùng random seed cố định và không làm thay đổi nội dung file.

In [ ]:
import random

if SOURCE_CHOICE == "ccpd":
    CCPD_URL = "https://github.com/Yvonne761/Chinese-Chess-Practical-Dataset.git"
    CCPD_REPO = Path("/content/Chinese-Chess-Practical-Dataset")
    CCPD_SUBDIR = "Dataset/對局/大師對局/以賽事分類/世界智力運動會"

    if not CCPD_REPO.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none",
            "--sparse", "--branch", "main", CCPD_URL, str(CCPD_REPO)
        ], check=True)
    else:
        remote = subprocess.check_output([
            "git", "-C", str(CCPD_REPO), "remote", "get-url", "origin"
        ], text=True).strip()
        if remote.rstrip("/").removesuffix(".git") != CCPD_URL.removesuffix(".git"):
            raise ValueError(f"Unexpected CCPD remote: {remote}")
        subprocess.run([
            "git", "-C", str(CCPD_REPO), "fetch", "--depth", "1", "origin", "main"
        ], check=True)
        subprocess.run(["git", "-C", str(CCPD_REPO), "switch", "main"], check=True)
        subprocess.run([
            "git", "-C", str(CCPD_REPO), "pull", "--ff-only", "origin", "main"
        ], check=True)

    subprocess.run([
        "git", "-C", str(CCPD_REPO), "sparse-checkout", "set",
        "--cone", CCPD_SUBDIR
    ], check=True)
    CCPD_SHA = subprocess.check_output([
        "git", "-C", str(CCPD_REPO), "rev-parse", "HEAD"
    ], text=True).strip()
    MATCH_DIR = CCPD_REPO / CCPD_SUBDIR
    ALL_PGN = sorted(MATCH_DIR.glob("*.pgn"))
    if not ALL_PGN:
        raise FileNotFoundError(f"No PGN files at {MATCH_DIR}")

    if CCPD_MODE == "sample":
        SELECTED_PGN = sorted(random.Random(CCPD_SEED).sample(
            ALL_PGN, min(CCPD_SAMPLE_SIZE, len(ALL_PGN))
        ))
        INGEST_INPUTS = SELECTED_PGN
    else:
        SELECTED_PGN = ALL_PGN
        INGEST_INPUTS = [MATCH_DIR]

    COPYRIGHT_FILES = [CCPD_REPO / "LICENSE", CCPD_REPO / "README.md"]
    if not all(p.is_file() for p in COPYRIGHT_FILES):
        raise FileNotFoundError("Missing CCPD LICENSE or README.md")
    INGEST_INPUTS = [*INGEST_INPUTS, *COPYRIGHT_FILES]
    VERIFY_FILES = [*SELECTED_PGN, *COPYRIGHT_FILES]
    SOURCE_NAME = "ccpd"
    SOURCE_VERSION = "git-" + CCPD_SHA
    SOURCE_LICENSE = "CC-BY-4.0"
    INCLUDE_PATTERNS = ["*.pgn", "LICENSE", "README.md"]
    print("CCPD SHA:", CCPD_SHA)
    print("PGN available:", len(ALL_PGN), "| selected:", len(SELECTED_PGN))
else:
    ALPHABETA_DIR = DRIVE_PROJECT / "datasets"
    if not ALPHABETA_DIR.is_dir():
        raise FileNotFoundError(ALPHABETA_DIR)
    VERIFY_FILES = sorted(ALPHABETA_DIR.glob("*.jsonl"))
    if not VERIFY_FILES:
        raise ValueError("No Alpha-Beta JSONL datasets found")
    INGEST_INPUTS = [ALPHABETA_DIR]
    SOURCE_NAME = "alphabeta"
    SOURCE_VERSION = "colab-batch-v1"
    SOURCE_LICENSE = None
    INCLUDE_PATTERNS = ["*.jsonl"]
    print("Alpha-Beta JSONL:", len(VERIFY_FILES))

## 4. Xem file PGN trước khi ingest

Một số PGN không phải UTF-8. Bronze chỉ lưu raw bytes; việc giải mã notation sẽ được thực hiện ở CCPD Silver Adapter.

In [ ]:
print("Selected files (first 10):")
for path in VERIFY_FILES[:10]:
    print("-", path.name, f"{path.stat().st_size:,} bytes")
if SOURCE_CHOICE == "ccpd":
    raw = SELECTED_PGN[0].read_bytes()
    print("First PGN bytes:", repr(raw[:100]))
    for codec in ("utf-8", "big5", "gb18030"):
        try:
            raw.decode(codec)
            print("Possible encoding:", codec)
            break
        except UnicodeDecodeError:
            continue
    else:
        print("Unknown text encoding; raw bytes are still preserved.")

## 5. Chạy BronzeIngester qua CLI

Tất cả file vào MyDrive/XiangqiAI/data/01_bronze/<source>. Re-run không ghi đè file có cùng hash; lỗi nếu cùng tên khác nội dung.

In [ ]:
command = [
    sys.executable, "-m", "scripts.ingest_bronze",
    *[str(p) for p in INGEST_INPUTS],
    "--destination-root", str(BRONZE_ROOT),
    "--source", SOURCE_NAME,
    "--source-version", SOURCE_VERSION,
    "--include", *INCLUDE_PATTERNS,
]
if SOURCE_LICENSE is not None:
    command.extend(["--license", SOURCE_LICENSE])
subprocess.run(command, cwd=BACKEND_DIR, check=True)

## 6. Đối chiếu SHA-256 với Drive

Kiểm tra mỗi file Bronze và manifest. Chỉ khi mọi hash khớp mới báo ingest hoàn tất.

In [ ]:
import hashlib
from src.data.bronze.models import BronzeFileManifest

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

verified = 0
for original in VERIFY_FILES:
    if original.stat().st_size == 0:
        continue
    copied = BRONZE_ROOT / SOURCE_NAME / original.name
    manifest_path = copied.with_name(copied.name + ".manifest.json")
    if not copied.is_file() or not manifest_path.is_file():
        raise FileNotFoundError(f"Incomplete Bronze output: {copied}")
    manifest = BronzeFileManifest.model_validate_json(
        manifest_path.read_text(encoding="utf-8")
    )
    checksum = sha256_file(original)
    if manifest.file_sha256 != checksum or sha256_file(copied) != checksum:
        raise AssertionError(f"SHA-256 mismatch: {copied}")
    if manifest.source != SOURCE_NAME:
        raise AssertionError(f"Source mismatch: {copied}")
    verified += 1

print("Verified:", verified, "files")
print("Bronze destination:", BRONZE_ROOT / SOURCE_NAME)

## Lưu ý sau khi chạy

- Nếu sample ổn, đổi CCPD_MODE="full" và chạy lại từ đầu. 20 PGN đã ingest trước sẽ được bỏ qua nếu hash không đổi.
- Toàn bộ bản gốc sẽ nằm trong XiangqiAI/data/01_bronze/ccpd; có kèm LICENSE, README.md và SHA-256 manifest cho từng file.
- Dataset PGN KHÔNG chạy được Silver Alpha-Beta hiện tại. Phải làm CCPDAdapter để giải mã PGN và replay GameEngine trước khi dùng cho huấn luyện.
- Nguồn CCPD và license: https://github.com/Yvonne761/Chinese-Chess-Practical-Dataset